<a href="https://colab.research.google.com/github/subhamsam007/nassau-candy-profitability-analysis/blob/main/Nassau_Candy_Product_Profitability_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("/content/Nassau Candy Distributor.csv")

In [2]:
print(df.shape)
df.head()

(5404, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2.0,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2.0,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3.0,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3.0,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3.0,7.35,3.90


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5404 entries, 0 to 5403
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          5404 non-null   int64  
 1   Order ID        5404 non-null   object 
 2   Order Date      5404 non-null   object 
 3   Ship Date       5404 non-null   object 
 4   Ship Mode       5404 non-null   object 
 5   Customer ID     5404 non-null   int64  
 6   Country/Region  5404 non-null   object 
 7   City            5404 non-null   object 
 8   State/Province  5404 non-null   object 
 9   Postal Code     5404 non-null   object 
 10  Division        5404 non-null   object 
 11  Region          5404 non-null   object 
 12  Product ID      5404 non-null   object 
 13  Product Name    5403 non-null   object 
 14  Sales           5403 non-null   float64
 15  Units           5403 non-null   float64
 16  Gross Profit    5403 non-null   float64
 17  Cost            5403 non-null   f

In [11]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Country/Region', 'City', 'State/Province',
       'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name',
       'Sales', 'Units', 'Gross Profit', 'Cost', 'calc_profit'],
      dtype='object')

In [4]:
df.isnull().sum()

,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Country/Region,0
City,0
State/Province,0
Postal Code,0


In [5]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce", dayfirst=True)
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce", dayfirst=True)

print(df[["Order Date", "Ship Date"]].isnull().sum())
print(df["Order Date"].min(), df["Order Date"].max())

Order Date    0
Ship Date     0
dtype: int64
2024-01-02 00:00:00 2025-08-15 00:00:00


Check duplicates

In [6]:
print(df.duplicated().sum())
print(df["Row ID"].duplicated().sum())

0
0


Check invalid values

In [7]:
print("Sales <= 0:", (df["Sales"] <= 0).sum())
print("Units <= 0:", (df["Units"] <= 0).sum())
print("Cost < 0:", (df["Cost"] < 0).sum())
print("Gross Profit < 0:", (df["Gross Profit"] < 0).sum())

Sales <= 0: 0
Units <= 0: 0
Cost < 0: 0
Gross Profit < 0: 0


. Validate that profit matches sales and cost

In [8]:
df["calc_profit"] = df["Sales"] - df["Cost"]
mismatch = (df["calc_profit"] - df["Gross Profit"]).abs() > 0.01
print("Profit mismatches:", mismatch.sum())

Profit mismatches: 0


Standardize labels

In [9]:
for col in ["Division", "Region", "Product Name", "Ship Mode", "State/Province", "City"]:
    df[col] = df[col].str.strip()

print(df["Division"].unique())
print(df["Region"].unique())
print(df["Product Name"].nunique())

['Chocolate' 'Other' 'Sugar']
['Interior' 'Atlantic' 'Gulf' 'Pacific']
15


In [12]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Country/Region', 'City', 'State/Province',
       'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name',
       'Sales', 'Units', 'Gross Profit', 'Cost', 'calc_profit'],
      dtype='object')

# Step 3: Profitability Metrics

The business problem is "sales ≠ profit". Right now you only have raw columns (Sales, Cost, Gross Profit). This step turns them into numbers that show how good each sale and each product is.

In [15]:
df["Margin %"] = (df["Gross Profit"] / df["Sales"]) * 100
df["Profit per Unit"] = df["Gross Profit"] / df["Units"]

Group by product

In [16]:
product = df.groupby(["Product Name", "Division"]).agg(
    Total_Sales=("Sales", "sum"),
    Total_Cost=("Cost", "sum"),
    Total_Profit=("Gross Profit", "sum"),
    Total_Units=("Units", "sum")
).reset_index()

Calculate metrics from the totals

In [17]:
product["Gross Margin %"] = (product["Total_Profit"] / product["Total_Sales"]) * 100
product["Profit per Unit"] = product["Total_Profit"] / product["Total_Units"]
product["Profit Contribution %"] = (product["Total_Profit"] / product["Total_Profit"].sum()) * 100
product["Sales Contribution %"] = (product["Total_Sales"] / product["Total_Sales"].sum()) * 100

In [18]:
product = product.round(2).sort_values("Total_Profit", ascending=False)
product

,Product Name,Division,Total_Sales,Total_Cost,Total_Profit,Total_Units,Gross Margin %,Profit per Unit,Profit Contribution %,Sales Contribution %
13,Wonka Bar -Scrumdiddlyumptious,Chocolate,14936.40,4563.90,10372.50,4149.0,69.44,2.50,21.07,19.99
12,Wonka Bar - Triple Dazzle Caramel,Chocolate,14958.75,5185.70,9773.05,3989.0,65.33,2.45,19.85,20.02
10,Wonka Bar - Milk Chocolate,Chocolate,14634.75,5133.42,9501.33,4503.0,64.92,2.11,19.30,19.59
9,Wonka Bar - Fudge Mallows,Chocolate,13143.60,4381.20,8762.40,3651.0,66.67,2.40,17.80,17.59
11,Wonka Bar - Nutty Crunch Surprise,Chocolate,12117.28,3472.00,8645.28,3472.0,71.35,2.49,17.56,16.22
6,Lickable Wallpaper,Other,3680.00,1840.00,1840.00,184.0,50.00,10.00,3.74,4.93
14,Wonka Gum,Other,316.25,151.80,164.45,253.0,52.00,0.65,0.33,0.42
4,Kazookles,Other,747.50,690.00,57.50,230.0,7.69,0.25,0.12,1.00
3,Hair Toffee,Sugar,45.00,10.00,35.00,10.0,77.78,3.50,0.07,0.06
0,Everlasting Gobstopper,Sugar,40.00,8.00,32.00,4.0,80.00,8.00,0.06,0.05


# Step 4: Product-Level Ranking & Classification

Rank products by profit and by margin

In [20]:
product["Profit Rank"] = product["Total_Profit"].rank(ascending=False, method="min").astype(int)
product["Margin Rank"] = product["Gross Margin %"].rank(ascending=False, method="min").astype(int)

In [21]:
sales_cut = product["Total_Sales"].median()
margin_cut = (product["Total_Profit"].sum() / product["Total_Sales"].sum()) * 100

print("Sales cut-off:", round(sales_cut, 2))
print("Margin cut-off:", round(margin_cut, 2))

Sales cut-off: 316.25
Margin cut-off: 65.9


Classify each product

In [22]:
def classify(row):
    high_sales = row["Total_Sales"] >= sales_cut
    high_margin = row["Gross Margin %"] >= margin_cut
    if high_sales and high_margin:
        return "Star: High Sales / High Margin"
    elif high_sales and not high_margin:
        return "Risk: High Sales / Low Margin"
    elif not high_sales and high_margin:
        return "Niche: Low Sales / High Margin"
    else:
        return "Weak: Low Sales / Low Margin"

product["Category"] = product.apply(classify, axis=1)

In [23]:
product[["Product Name", "Division", "Total_Sales", "Total_Profit",
         "Gross Margin %", "Profit Rank", "Margin Rank", "Category"]].sort_values("Profit Rank")

,Product Name,Division,Total_Sales,Total_Profit,Gross Margin %,Profit Rank,Margin Rank,Category
13,Wonka Bar -Scrumdiddlyumptious,Chocolate,14936.40,10372.50,69.44,1,4,Star: High Sales / High Margin
12,Wonka Bar - Triple Dazzle Caramel,Chocolate,14958.75,9773.05,65.33,2,6,Risk: High Sales / Low Margin
10,Wonka Bar - Milk Chocolate,Chocolate,14634.75,9501.33,64.92,3,7,Risk: High Sales / Low Margin
9,Wonka Bar - Fudge Mallows,Chocolate,13143.60,8762.40,66.67,4,5,Star: High Sales / High Margin
11,Wonka Bar - Nutty Crunch Surprise,Chocolate,12117.28,8645.28,71.35,5,3,Star: High Sales / High Margin
6,Lickable Wallpaper,Other,3680.00,1840.00,50.00,6,11,Risk: High Sales / Low Margin
14,Wonka Gum,Other,316.25,164.45,52.00,7,10,Risk: High Sales / Low Margin
4,Kazookles,Other,747.50,57.50,7.69,8,15,Risk: High Sales / Low Margin
3,Hair Toffee,Sugar,45.00,35.00,77.78,9,2,Niche: Low Sales / High Margin
0,Everlasting Gobstopper,Sugar,40.00,32.00,80.00,10,1,Niche: Low Sales / High Margin
